# Multiple KCs and Column Mapping

Real datasets have many KCs and their own column names. This example uses simulated data laid out like an ASSISTments export and covers:

1. Mapping dataset columns with `column_mapping`.
2. Fitting many KCs, concurrently, and adding KCs later.
3. Per KC priors and refitting with `overwrite_kcs`.

See the [Data Format](../guide/data_format.rst) and [Memory and Performance](../guide/performance.rst) guide pages for details.

In [1]:
import logging

import numpy as np
import pandas as pd
from cmdstanpy.utils import get_logger

# keep the notebook output readable, CmdStan progress is logged at INFO level
get_logger().setLevel(logging.WARNING)

## An ASSISTments style dataset

We rename the simulated columns to the names an ASSISTments skill builder export uses: `user_id`, `problem_id`, `skill_id`, `correct` and `order_id`. To use a real export, load it with `pd.read_csv` and keep these columns.

In [2]:
from stanbkt.utils import sim_simple_BKT

# problems are spread over the KCs, so 90 problems is about 15 per KC per student
n_kcs = 6
true_learn = np.linspace(0.05, 0.3, n_kcs).round(2)
sim = sim_simple_BKT(
    n_students=60, n_problems=90, n_kcs=n_kcs, rng_seed=41, frac=0.8,
    learn=list(true_learn), guess=0.2, slip=0.1,
)
assist_df = sim.rename(columns={"student_id": "user_id", "kc_id": "skill_id", "timestamp": "order_id"})
assist_df["order_id"] = assist_df.groupby("user_id")["order_id"].rank(method="first").astype(int)
assist_df.head()

,user_id,problem_id,correct,order_id,skill_id
0,stu_0,prob_1,0,1,kc_5
1,stu_0,prob_2,1,2,kc_4
2,stu_0,prob_4,0,3,kc_4
3,stu_0,prob_5,0,4,kc_0
4,stu_0,prob_7,1,5,kc_4


## Map the columns

`column_mapping` maps StanBKT's `ColumnNames` to your column names. Only the columns that differ need an entry, and the same mapping is passed to `fit`, every `predict*` method and `summary`.

In [3]:
from stanbkt.utils import ColumnNames

col_mapping = {
    ColumnNames.STUDENT_ID: "user_id",
    ColumnNames.KC_ID: "skill_id",
    ColumnNames.ORDER: "order_id",
}

## Fit many KCs

KCs are independent fits, so `fit` runs several at once. With the default `n_kcs_workers=-1`, the number of concurrent KCs is the available CPUs divided by the cores one fit uses (`parallel_chains` for MCMC), capped by the number of KCs.

In [4]:
import time

from stanbkt.fits import MCMCFitOptions
from stanbkt.models import StandardBKT
from stanbkt.utils import VerbosityLevel

fit_opts = MCMCFitOptions(seed=1234, iter_warmup=500, iter_sampling=500, show_progress=False)
first_kcs = ["kc_0", "kc_1", "kc_2", "kc_3"]

model = StandardBKT(verbose=VerbosityLevel.WARN)
start = time.perf_counter()
model.fit(assist_df[assist_df["skill_id"].isin(first_kcs)], column_mapping=col_mapping,
          stan_fit_options=fit_opts)
print(f"fitted {model.fits.num_fitted_kcs} KCs in {time.perf_counter() - start:.1f}s")

fitted 4 KCs in 0.8s


Fitting is additive: calling `fit` with new KCs adds them to the model.

In [5]:
model.fit(assist_df[~assist_df["skill_id"].isin(first_kcs)], column_mapping=col_mapping,
          stan_fit_options=fit_opts)
sorted(model.fits.get_fitted_kcs())

['kc_0', 'kc_1', 'kc_2', 'kc_3', 'kc_4', 'kc_5']

In [6]:
learn = model.summary(column_mapping=col_mapping).xs("learn", level="parameter")
learn = learn[["Mean", "2.5%", "97.5%", "R_hat"]].round(3)
learn["truth"] = true_learn
learn

,,,Mean,2.5%,97.5%,R_hat,truth
skill_id,axis,label,,,,,
kc_0,group_id,1,0.066,0.028,0.113,1.009,0.05
kc_1,group_id,1,0.082,0.049,0.124,1.004,0.10
kc_2,group_id,1,0.164,0.115,0.224,1.002,0.15
kc_3,group_id,1,0.189,0.116,0.273,1.002,0.20
kc_4,group_id,1,0.292,0.206,0.388,1.003,0.25
kc_5,group_id,1,0.335,0.245,0.443,1.004,0.30


## Per KC priors and refitting

Refitting a KC that already has a fit raises an error, to avoid overwriting results by accident. Pass `overwrite_kcs=True` to refit. Here we refit two KCs with their own priors, passed as a dict keyed by KC ID.

In [7]:
from stanbkt.models import StandardPriors

try:
    model.fit(assist_df[assist_df["skill_id"] == "kc_0"], column_mapping=col_mapping)
except ValueError as e:
    print("ValueError:", str(e)[:100], "...")

ValueError: Fit for KC 'kc_0' already exists. Set 'overwrite_kcs=True' to overwrite. ...


In [8]:
refit_priors = {
    "kc_0": StandardPriors(learn_mu=-2.5, learn_std=0.5),
    "kc_5": StandardPriors(learn_mu=-1.0, learn_std=0.5),
}
refit_df = assist_df[assist_df["skill_id"].isin(refit_priors)]
model.fit(refit_df, column_mapping=col_mapping, priors=refit_priors,
          stan_fit_options=fit_opts, overwrite_kcs=True)
model.summary(kcs=["kc_0", "kc_5"], column_mapping=col_mapping).xs("learn", level="parameter")[["Mean", "2.5%", "97.5%"]]

,,,Mean,2.5%,97.5%
skill_id,axis,label,,,
kc_0,group_id,1,0.072227,0.040303,0.114955
kc_5,group_id,1,0.325854,0.243693,0.419649


## Predict

`predict` returns one row per interaction for every fitted KC in the data, with the column names from `column_mapping`. The same holds for every `predict*` method.

In [9]:
preds = model.predict(assist_df, column_mapping=col_mapping)
preds.head()

,skill_id,user_id,problem_id,pKnow,pCorrectness,correct
0,kc_5,stu_0,prob_1,0.275624,0.351263,0
1,kc_5,stu_0,prob_9,0.353922,0.409983,0
2,kc_5,stu_0,prob_20,0.365483,0.418653,0
3,kc_5,stu_0,prob_23,0.367387,0.420082,0
4,kc_5,stu_0,prob_31,0.367707,0.420321,1
